In [1]:
import pandas as pd
from influxdb_client import InfluxDBClient

In [ ]:
# =====================================================================
# 1. DATABASE CONNECTION CONFIGURATION
# url = "http://localhost:8086"
url = "https://influx.htetarkar.online/"
token = "TxfwaNhZSU4C-e_F0lMU9fPty2JegGPS9bxEzjqabwozmejkazMH8Ch_Tts7-anAN17jBGHKtlMThhz4bw3g_A=="
org = "essco"
bucket = "essco_energy_data"

# Initialize the official InfluxDB client
client = InfluxDBClient(url=url, token=token, org=org)
query_api = client.query_api()

In [3]:
# =====================================================================
# 2. THE FLUX QUERY (Pivoting into columns natively)
# =====================================================================
flux_query = f'''
from(bucket: "{bucket}")
  |> range(start: -1h)
  |> filter(fn: (r) => r["_measurement"] == "inverter")
  |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
'''

print("Querying InfluxDB...")
df = query_api.query_data_frame(flux_query)
df

Querying InfluxDB...


,result,table,_start,_stop,_time,_measurement,deviceId,gateway_did,model,protocol_std,...,pv_to_inverter_active_status,reactive_power_control_mode,reactive_power_setpoint_var,solar_generation_accumulated_wh,solar_string_1_power_watts,solar_string_1_voltage,solar_string_2_power_watts,solar_string_2_voltage,system_operation_state_code,transmission_interval_sec
0,_result,0,2026-06-10 07:38:35.153999+00:00,2026-06-10 08:38:35.153999+00:00,2026-06-10 07:39:11+00:00,inverter,4,1,AT50,1,...,0.0,0.0,0.0,28978400.0,2017.0,364.1,1923.0,335.20,3.0,60.0
1,_result,0,2026-06-10 07:38:35.153999+00:00,2026-06-10 08:38:35.153999+00:00,2026-06-10 07:40:11+00:00,inverter,4,1,AT50,1,...,0.0,0.0,0.0,28978400.0,1878.0,362.4,1780.0,336.80,3.0,60.0
2,_result,0,2026-06-10 07:38:35.153999+00:00,2026-06-10 08:38:35.153999+00:00,2026-06-10 07:41:11+00:00,inverter,4,1,AT50,1,...,0.0,0.0,0.0,28978500.0,2403.0,363.0,2273.0,332.90,3.0,60.0
3,_result,0,2026-06-10 07:38:35.153999+00:00,2026-06-10 08:38:35.153999+00:00,2026-06-10 07:42:11+00:00,inverter,4,1,AT50,1,...,0.0,0.0,0.0,28978600.0,2731.0,367.9,2569.0,332.80,3.0,60.0
4,_result,0,2026-06-10 07:38:35.153999+00:00,2026-06-10 08:38:35.153999+00:00,2026-06-10 07:43:11+00:00,inverter,4,1,AT50,1,...,0.0,0.0,0.0,28978700.0,3163.0,361.8,2984.0,329.40,3.0,60.0
5,_result,0,2026-06-10 07:38:35.153999+00:00,2026-06-10 08:38:35.153999+00:00,2026-06-10 07:44:11+00:00,inverter,4,1,AT50,1,...,0.0,0.0,0.0,28978800.0,3414.0,365.2,3199.0,329.80,3.0,60.0
6,_result,0,2026-06-10 07:38:35.153999+00:00,2026-06-10 08:38:35.153999+00:00,2026-06-10 07:45:11+00:00,inverter,4,1,AT50,1,...,0.0,0.0,0.0,28978900.0,3105.0,358.2,2976.0,326.00,3.0,60.0
7,_result,0,2026-06-10 07:38:35.153999+00:00,2026-06-10 08:38:35.153999+00:00,2026-06-10 07:46:11+00:00,inverter,4,1,AT50,1,...,0.0,0.0,0.0,28979000.0,3160.0,360.4,2967.0,331.60,3.0,60.0
8,_result,0,2026-06-10 07:38:35.153999+00:00,2026-06-10 08:38:35.153999+00:00,2026-06-10 07:47:11+00:00,inverter,4,1,AT50,1,...,0.0,0.0,0.0,28979000.0,2679.0,360.2,2483.0,320.10,3.0,60.0
9,_result,0,2026-06-10 07:38:35.153999+00:00,2026-06-10 08:38:35.153999+00:00,2026-06-10 07:48:11+00:00,inverter,4,1,AT50,1,...,0.0,0.0,0.0,28979100.0,1642.0,351.6,1623.0,326.60,3.0,60.0


In [22]:
# =====================================================================
# 3. TIME-ZONE CONVERSION & CLEANING (Thailand Time)
# =====================================================================
if not df.empty:
    # Drop internal InfluxDB system column overhead to keep our matrix tidy
    columns_to_drop = ['result', 'table', '_start', '_stop', '_measurement']
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
    
    # Step A: Explicitly ensure Python reads the timestamp as UTC
    df['_time'] = pd.to_datetime(df['_time'], utc=True)
    
    # Step B: Shift the entire timeline directly to Bangkok local time (+7)
    df['local_time'] = df['_time'].dt.tz_convert('Asia/Bangkok')
    
    # Move 'local_time' to the front of the DataFrame layout for readability
    cols = ['local_time'] + [col for col in df.columns if col != 'local_time' and col != '_time']
    df = df[cols]
    
    print("Data successfully retrieved and shifted to Thailand Time!")
else:
    print("No data returned. Check your token, bucket name, or timeframe range.")

# Display the top 5 rows of your structured matrix
df.head()

Data successfully retrieved and shifted to Thailand Time!


,local_time,deviceId,gateway_did,model,protocol_std,siteId,ac_power_phase_a,ac_power_phase_b,ac_power_phase_c,ac_voltage_phase_a,...,pv_to_inverter_active_status,reactive_power_control_mode,reactive_power_setpoint_var,solar_generation_accumulated_wh,solar_string_1_power_watts,solar_string_1_voltage,solar_string_2_power_watts,solar_string_2_voltage,system_operation_state_code,transmission_interval_sec
0,2026-05-29 14:06:10+07:00,4,1,AT50,1,ESSCO_49453AB5,994.56,1005.06,998.46,236.8,...,0.0,0.0,0.0,28535400.0,1579.0,371.7,1550.0,340.7,3.0,60.0
1,2026-05-29 14:07:10+07:00,4,1,AT50,1,ESSCO_49453AB5,1159.83,1148.16,1140.72,236.7,...,0.0,0.0,0.0,28535500.0,1813.0,371.2,1779.0,337.3,3.0,60.0
2,2026-05-29 14:08:10+07:00,4,1,AT50,1,ESSCO_49453AB5,1136.64,1148.16,1117.44,236.8,...,0.0,0.0,0.0,28535600.0,1753.0,361.6,1693.0,331.5,3.0,60.0
3,2026-05-29 14:09:11+07:00,4,1,AT50,1,ESSCO_49453AB5,1207.68,1196.00,1187.28,236.8,...,0.0,0.0,0.0,28535600.0,1869.0,367.9,1802.0,330.8,3.0,60.0
4,2026-05-29 14:10:10+07:00,4,1,AT50,1,ESSCO_49453AB5,2039.06,2035.75,2005.52,237.1,...,0.0,0.0,0.0,28535700.0,3209.0,366.0,3118.0,342.7,3.0,60.0


In [24]:
# =====================================================================
# 1. TIME RESAMPLING (Handling Text and Numbers Separately)
# =====================================================================
# Set the 'local_time' column as the index
df_resampled = df.set_index('local_time')

# Separate numeric columns from text columns
numeric_cols = df_resampled.select_dtypes(include=['number']).columns
text_cols = df_resampled.select_dtypes(exclude=['number']).columns

# Create the uniform 5-minute grid time-steps
# A. For numbers: Take the average if multiple points exist, then forward-fill gaps
df_numeric = df_resampled[numeric_cols].resample('5min').mean().ffill()

# B. For text tags: Just take the first available string in that window, then forward-fill gaps
df_text = df_resampled[text_cols].resample('5min').first().ffill()

# Combine them back together horizontally into a single clean grid
df_final = pd.concat([df_numeric, df_text], axis=1).reset_index()

# Round numeric columns to 2 decimal places for cleaner display
df_final[numeric_cols] = df_final[numeric_cols].round(2)

# =====================================================================
# 2. FEATURE ENGINEERING (Extracting AI behavioral variables)
# =====================================================================
df_final['hour_of_day'] = df_final['local_time'].dt.hour
df_final['day_of_week'] = df_final['local_time'].dt.dayofweek

# Move our structural tracking features to the front for clean layout viewing
front_cols = ['local_time', 'hour_of_day', 'day_of_week', 'siteId', 'model', 'deviceId']
remaining_cols = [c for c in df_final.columns if c not in front_cols]
df_final = df_final[front_cols + remaining_cols]

print("Pristine, Uniform AI Sequence Matrix Generated Successfully!")
df_final.head()

Pristine, Uniform AI Sequence Matrix Generated Successfully!


,local_time,hour_of_day,day_of_week,siteId,model,deviceId,ac_power_phase_a,ac_power_phase_b,ac_power_phase_c,ac_voltage_phase_a,...,reactive_power_setpoint_var,solar_generation_accumulated_wh,solar_string_1_power_watts,solar_string_1_voltage,solar_string_2_power_watts,solar_string_2_voltage,system_operation_state_code,transmission_interval_sec,gateway_did,protocol_std
0,2026-05-29 14:05:00+07:00,14,4,ESSCO_49453AB5,AT50,4,1124.68,1124.34,1110.97,236.78,...,0.0,28535525.0,1753.5,368.10,1706.0,335.08,3.0,60.0,1,1
1,2026-05-29 14:10:00+07:00,14,4,ESSCO_49453AB5,AT50,4,1676.67,1670.40,1649.22,236.78,...,0.0,28535860.0,2645.2,365.60,2530.2,334.48,3.0,60.0,1,1
2,2026-05-29 14:15:00+07:00,14,4,ESSCO_49453AB5,AT50,4,937.34,936.13,925.61,236.70,...,0.0,28536180.0,1449.2,361.32,1417.6,331.66,3.0,60.0,1,1
3,2026-05-29 14:20:00+07:00,14,4,ESSCO_49453AB5,AT50,4,1060.79,1050.29,1041.67,236.78,...,0.0,28536420.0,1636.4,365.42,1587.0,334.92,3.0,60.0,1,1
4,2026-05-29 14:25:00+07:00,14,4,ESSCO_49453AB5,AT50,4,1708.74,1694.84,1683.38,237.22,...,0.0,28536720.0,2657.4,363.66,2556.6,336.78,3.0,60.0,1,1


In [14]:
# =====================================================================
# 2. THE FLUX QUERY (Pivoting into columns natively)
# =====================================================================
# This script pulls the last 1 hour of inverter data and collapses
# your descriptive rows into readable horizontal spreadsheet columns.
flux_query = f'''
from(bucket: "{bucket}")
  |> range(start: -2h)
  |> filter(fn: (r) => r["_measurement"] == "weather")
  |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
'''

print("Querying InfluxDB...")
# Execute query and stream data straight into a Pandas DataFrame
df = query_api.query_data_frame(flux_query)
df  # Display the first few rows of the DataFrame

Querying InfluxDB...


,result,table,_start,_stop,_time,_measurement,siteId,ambient_temp,cloud_cover,humidity,solar_irradiance
0,_result,0,2026-05-29 05:07:48.229558+00:00,2026-05-29 07:07:48.229558+00:00,2026-05-29 05:45:00+00:00,weather,ESSCO,35.2,100.0,51.0,755.0
1,_result,0,2026-05-29 05:07:48.229558+00:00,2026-05-29 07:07:48.229558+00:00,2026-05-29 06:00:00+00:00,weather,ESSCO,35.2,100.0,51.0,733.0
2,_result,0,2026-05-29 05:07:48.229558+00:00,2026-05-29 07:07:48.229558+00:00,2026-05-29 06:30:00+00:00,weather,ESSCO,35.0,89.0,53.0,649.0
3,_result,0,2026-05-29 05:07:48.229558+00:00,2026-05-29 07:07:48.229558+00:00,2026-05-29 06:45:00+00:00,weather,ESSCO,34.6,96.0,53.0,589.0
4,_result,0,2026-05-29 05:07:48.229558+00:00,2026-05-29 07:07:48.229558+00:00,2026-05-29 07:00:00+00:00,weather,ESSCO,35.7,100.0,50.0,821.0
